# Final experiments, solo run (Group 19)

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom. Every training run saves its own result file and is **skipped if it already exists**, so after a disconnect just run the setup cells again and continue where you stopped.

Order: 0 setup → 1 quick check → 2 main study → 3 size check → 4 new backbones → 5 final test (hospital 2, once) → 6 ablations → 7 leave-one-hospital-out → 8 size from models → 9 download results.

Settings are frozen: σ = 0.2, pooled reference, 5 epochs, 40,000 patches, threshold from id_val.

## 0. Setup (run these every time you open a new session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE   = "/content/drive/MyDrive/ai_project_widad"   # your own folder; ai-project-final.zip must be in it
RESULTS = f"{DRIVE}/results"; CKPT = f"{DRIVE}/ckpt"; CACHE = f"{DRIVE}/cache"
FINAL   = f"{DRIVE}/final";   SIZE = f"{DRIVE}/size"; SUMMARY = f"{DRIVE}/summary"; WEIGHTS = f"{DRIVE}/weights"
STAGE   = f"{DRIVE}/stage_labels.csv"
MAIN_TAG = "main"
SEEDS = "0 1 2"
for d in (RESULTS, CKPT, CACHE, FINAL, SIZE, SUMMARY, WEIGHTS):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{DRIVE}/ai-project-final.zip"), f"put ai-project-final.zip in {DRIVE}"

In [ ]:
!unzip -q -o "$DRIVE/ai-project-final.zip" -d /content
%cd /content/ai-project
!pip -q install wilds scikit-image opencv-python-headless pyyaml tqdm
!python -m pytest tests -q 2>&1 | tail -2

In [ ]:
# Data: about 10 GB, 5-10 minutes, needed in every new session (it lives on the session's disk, not your Drive)
from wilds import get_dataset
ds = get_dataset(dataset='camelyon17', download=True, root_dir='/content/data')
print(len(ds), "patches; metadata columns:", list(ds._metadata_df.columns))

In [ ]:
# Minutes per run on this GPU (only needed once)
!python scripts/speed_test.py --backbones resnet18 resnet18_histo resnet50_histo

## 1. Quick check on real data (about 10 minutes)

A small, fast run that only checks the pipeline works on the real data. Its numbers are rough (1 epoch, 4,000 patches) and are saved outside your results folder.

**What to look for:** id_val accuracy clearly higher than ood_val (hospital 1) accuracy, and the baseline with high sensitivity but lower specificity on ood_val, as in the interim results.

In [ ]:
!python scripts/run_experiments.py --root /content/data --cache /content/cache_check --out /content/check --n_train 4000 --n_eval 2000 --epochs 1 --seeds 0 --methods baseline vahadane --tag check

## 2. Main study with checkpoints (20 runs: 4 methods × 5 seeds)

This repeats the interim study on your own account and saves the model weights the final test needs.

**Check against the interim results** (hospital 1, mean ± SD over 5 seeds). Yours should land within about a point:

| Method | Accuracy | Sensitivity | Specificity | AUC |
|---|---|---|---|---|
| baseline | 0.852 ± 0.010 | 0.959 ± 0.013 | 0.746 ± 0.031 | 0.959 ± 0.003 |
| hed | 0.864 ± 0.005 | 0.935 ± 0.015 | 0.794 ± 0.019 | 0.956 ± 0.006 |
| vahadane | 0.884 ± 0.004 | 0.839 ± 0.007 | 0.930 ± 0.007 | 0.954 ± 0.005 |
| both | 0.872 ± 0.012 | 0.786 ± 0.025 | 0.958 ± 0.007 | 0.946 ± 0.007 |

In [ ]:
!python scripts/run_experiments.py --root /content/data --cache "$CACHE" --out "$RESULTS" --seeds 0 1 2 3 4 --tag main --ckpt_dir "$CKPT"

## 3. Can we measure metastasis size from patches? (about 2 minutes, no training)

Needs CAMELYON17's `stage_labels.csv` (18 kB). If the download fails, the cell lists the files on the public CAMELYON server so you can copy the right path.

In [ ]:
import urllib.request
if not os.path.exists(STAGE):
    try:
        urllib.request.urlretrieve("https://camelyon-dataset.s3.amazonaws.com/CAMELYON17/stages.csv", STAGE)
    except Exception as e:
        print("direct download failed:", e)
        !pip -q install awscli
        !aws s3 ls --no-sign-request s3://camelyon-dataset/CAMELYON17/training/
        # then: !aws s3 cp --no-sign-request s3://camelyon-dataset/<path shown above>/stage_labels.csv "$STAGE"
print(open(STAGE).read()[:200] if os.path.exists(STAGE) else f"Not found: put stage_labels.csv at {STAGE}")

In [ ]:
!python scripts/metastasis_size.py --root /content/data --stage_labels "$STAGE" --out "$SIZE"

## 4. New backbones (24 runs: 2 models × 4 methods × 3 seeds)
- `resnet18_histo`: the same ResNet-18, pretrained on 57 histology datasets (Ciga et al. 2022).
- `resnet50_histo`: ResNet-50 pretrained on about 33 million tissue patches (Kang et al. 2023). Slower per run.

In [ ]:
import urllib.request, sys
sys.path.insert(0, "/content/ai-project")
from src.models import WEIGHT_URLS
W18 = f"{WEIGHTS}/tenpercent_resnet18.ckpt"; W50 = f"{WEIGHTS}/mocov2_rn50_ep200.torch"
for bb, path in (("resnet18_histo", W18), ("resnet50_histo", W50)):
    if not os.path.exists(path):
        urllib.request.urlretrieve(WEIGHT_URLS[bb], path)
    print(bb, os.path.getsize(path) // 2**20, "MB")

In [ ]:
!python scripts/run_experiments.py --root /content/data --cache "$CACHE" --out "$RESULTS" --seeds {SEEDS} --backbone resnet18_histo --weights_path "$W18" --tag bb_r18histo --ckpt_dir "$CKPT"

In [ ]:
!python scripts/run_experiments.py --root /content/data --cache "$CACHE" --out "$RESULTS" --seeds {SEEDS} --backbone resnet50_histo --weights_path "$W50" --tag bb_r50histo --ckpt_dir "$CKPT"

## 5. Final evaluation

5a scores every patch of hospital 1. **5b is the one-time test on hospital 2.** Run it once, only after sections 2 and 4 have fully finished, and don't change any setting afterwards.

In [ ]:
TAGS = f"{MAIN_TAG} bb_r18histo bb_r50histo"
!python scripts/predict_hospital.py --root /content/data --cache "$CACHE" --results "$RESULTS" --ckpt_dir "$CKPT" --tags {TAGS} --hospital 1 --out "$FINAL"

In [ ]:
# ONE TIME ONLY
!python scripts/predict_hospital.py --root /content/data --cache "$CACHE" --results "$RESULTS" --ckpt_dir "$CKPT" --tags {TAGS} --hospital 2 --out "$FINAL" --unseal_test

## 6. Ablations on hospital 1
6a: HED strength σ = 0.05, 0.1 and 0.4 for HED and Both (18 runs). 6b: normalise to one lab's typical patch, hospitals 0, 3, 4 (9 runs; its patch copies go to the session disk to save Drive space).

In [ ]:
for s in ["0.05", "0.1", "0.4"]:
    !python scripts/run_experiments.py --root /content/data --cache "$CACHE" --out "$RESULTS" --methods hed both --sigma {s} --seeds {SEEDS} --tag abl_sigma{s}

In [ ]:
for r in ["h0", "h3", "h4"]:
    !python scripts/run_experiments.py --root /content/data --cache /content/cache_abl --out "$RESULTS" --methods vahadane --ref {r} --seeds {SEEDS} --tag abl_ref_{r}

## 7. Leave-one-hospital-out (60 runs: 5 folds × 4 methods × 3 seeds)

The longest part. If GPU time runs short, run it first with `LOHO_SEEDS = "0"` (20 runs) and add seeds 1 and 2 later: finished runs are skipped.

In [ ]:
LOHO_SEEDS = SEEDS
for h in [0, 1, 3, 4, 2]:
    !python scripts/run_experiments.py --root /content/data --cache /content/cache_loho --out "$RESULTS" --split loho --test_hospital {h} --unseal_test --seeds {LOHO_SEEDS} --tag loho_h{h}
    !rm -f /content/cache_loho/loho{h}_*

## 8. Metastasis size from each model's tumour calls (after section 5)

In [ ]:
!python scripts/metastasis_size.py --root /content/data --stage_labels "$STAGE" --final "$FINAL" --hospitals 1 2 --out "$SIZE"

## 9. Download the results (small tables only, no model weights)

In [ ]:
!python scripts/summarize.py --results "$RESULTS" --out "$SUMMARY"
!cd "$DRIVE" && zip -q -r /content/send_back.zip results summary size final/*.csv
from google.colab import files
files.download("/content/send_back.zip")